In [116]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 100000
learning_rate = 1e-4
eval_iters = 2500

cpu


In [117]:
chars = sorted(set(text))
print(chars)
print(len(chars))
vocab_size = len(chars)

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']
92


In [118]:
chars = sorted(set(text))
print(chars)
print(len(chars))

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']
92


In [119]:
with open('wizard of oz.txt', 'r' , encoding='utf-8')as f:
    text=f.read()
print((text[:200]))



  DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NEW


In [120]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([ 0,  1,  1, 33, 44, 47, 44, 49, 37, 54,  1, 30, 43, 33,  1, 49, 37, 34,
         1, 52, 38, 55, 30, 47, 33,  1, 38, 43,  1, 44, 55,  0,  0,  1,  1, 31,
        54,  0,  0,  1,  1, 41, 15,  1, 35, 47, 30, 43, 40,  1, 31, 30, 50, 42,
         0,  0,  1,  1, 30, 50, 49, 37, 44, 47,  1, 44, 35,  1, 49, 37, 34,  1,
        52, 38, 55, 30, 47, 33,  1, 44, 35,  1, 44, 55, 13,  1, 49, 37, 34,  1,
        41, 30, 43, 33,  1, 44, 35,  1, 44, 55])


In [121]:
n = int(0.8*len(data))
train_data= data[:n]
val_data = data[n:]
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)


inputs:
tensor([[ 1, 78, 73,  0, 77, 61, 59, 78],
        [59, 79, 65, 66, 78,  0, 66, 67],
        [73, 64,  1, 73, 59, 78, 71, 63],
        [52, 73, 65, 65, 70, 63, 14, 31]])
targets:
tensor([[78, 73,  0, 77, 61, 59, 78, 78],
        [79, 65, 66, 78,  0, 66, 67, 71],
        [64,  1, 73, 59, 78, 71, 63, 59],
        [73, 65, 65, 70, 63, 14, 31, 79]])


In [122]:
block_size = 8
x=train_data[:block_size]
y=val_data[1:block_size+1]

for t in range (block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is ', context, 'target is ', target)


when input is  tensor([0]) target is  tensor(73)
when input is  tensor([0, 1]) target is  tensor(1)
when input is  tensor([0, 1, 1]) target is  tensor(78)
when input is  tensor([ 0,  1,  1, 33]) target is  tensor(66)
when input is  tensor([ 0,  1,  1, 33, 44]) target is  tensor(63)
when input is  tensor([ 0,  1,  1, 33, 44, 47]) target is  tensor(1)
when input is  tensor([ 0,  1,  1, 33, 44, 47, 44]) target is  tensor(74)
when input is  tensor([ 0,  1,  1, 33, 44, 47, 44, 49]) target is  tensor(59)


In [123]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [124]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


dPB0'$Tq.J+‘h%QqN“qGuKv—XSz”F/booNI8;/E$pmq&Eml#]T,e”V&fX—e6’“m8_iL0N's?/;]M*Ps.a]Sf88O•™a(+wclN_rVHCZ1$buI]iCa6™™L0‘h)M[cCN0z"E;Qef1-;+‘ht ,qK(Nf1M2k•+rHHC' •l•!eD0-z—?!uV-G3z”t—iPm6?SA"$I[b!PsnqGU3 E6$&BaSU•o#aMO-olZUxNf&*Ae_M[+ujM"$[mWfShO%‘mQERJvVZtlt“VJZ* W™L[iO*™gv•ZLa6E]E9—5Bh2H™hfT+xbnMT&f_,ad;1U
S%
?X/uBx7#HH:v—/x&::/™/uRAbAc2$O;‘;ceZ]dcRQL)‘h;cwmkT—RQxZ1-W?C—HNR634;"Lx5VGhf3tZ3
(a(4uOu4;o3tj
H3%(MPnrr8w/6CD1B!Pkz”t49#iSmH—E%j9InqHXeU—“yt5ZjA5yMI
5:(RwWiOOWh./?6E$‘J#pj]
H$:?NfmU—_,-w+3Z


In [125]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range (max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
        
    xb, yb = get_batch('train')
    logits, loss= model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())
    

step: 0, train loss: 5.214, val loss: 5.205
step: 2500, train loss: 4.991, val loss: 4.988
step: 5000, train loss: 4.779, val loss: 4.794
step: 7500, train loss: 4.581, val loss: 4.609
step: 10000, train loss: 4.392, val loss: 4.424
step: 12500, train loss: 4.214, val loss: 4.259
step: 15000, train loss: 4.050, val loss: 4.105
step: 17500, train loss: 3.900, val loss: 3.952
step: 20000, train loss: 3.756, val loss: 3.822
step: 22500, train loss: 3.622, val loss: 3.699
step: 25000, train loss: 3.495, val loss: 3.584
step: 27500, train loss: 3.388, val loss: 3.483
step: 30000, train loss: 3.291, val loss: 3.381
step: 32500, train loss: 3.198, val loss: 3.297
step: 35000, train loss: 3.106, val loss: 3.222
step: 37500, train loss: 3.032, val loss: 3.163
step: 40000, train loss: 2.975, val loss: 3.099
step: 42500, train loss: 2.914, val loss: 3.041
step: 45000, train loss: 2.865, val loss: 2.993
step: 47500, train loss: 2.825, val loss: 2.964
step: 50000, train loss: 2.786, val loss: 2.928

In [127]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=1000)[0].tolist())
print(generated_chars)


TIld, t lused wan tlet d ch, D T p, hotoug asoutlswind ar vooure. ilallis  _DEAPdms Id bolve woon the, the

oun he t
"Wimsplilit'He teldod e'mithe fl s o ghed udegras o igoway brithain the ind wimepps
"Weve  ledgl he  so ichey f, m t wa.THQwe Ithouzpavear no me f thed on W('1Uucligsked s incascastodind

an, a, t ly anoom, impra

osthe jMU(unee_onde o " y wacto;ke fanoceyense
lepau, tum;QZ’$a re she c, sthe o t m han vin we!]s
hensisang-zLrople "Prs nekly, anfradis I u, e weancandlot PG2V!ir ie IEm anvitodims  wat al

"
winquthe sof To d cinngoket obllveleean'"HJ

m "
we
l t Bb ty ON29GDueat here boul,
s'rs,"HEm J?DO8wl grce, g, mie the  t t."f be grushinomy coun wsinowe th casix%Y™—“—W-winy bbllib'sedofuathetirngsasive os agl y, TThthend wlakedmimandn'tf opeanis?™CYAuntfatires sounoug, hengy TH1AFIAYNpplid tomungas; th lkeay p2y nengheaifoulusk 19or.
Ho,"$z, had puece the ckld s j
f TUV$YNN?Ystl aclimboucovenerpYS.
ce ceknst. rous orelachizaik, wouteanet y, m.
Calug,"
ngokeak tecant a